In [ ]:
from datasets import load_dataset
dataset = load_dataset("edinburghcstr/ami", "ihm")


In [ ]:
dataset


In [ ]:
dataset["train"][1]


In [ ]:
# AMI Dataset Statistics Analysis
import pandas as pd
from collections import defaultdict, Counter
import numpy as np

def analyze_ami_dataset(dataset):
    """Analyze AMI dataset statistics"""
    
    print("="*80)
    print("AMI DATASET STATISTICS")
    print("="*80)
    
    # General statistics by splits
    print("\n📊 GENERAL STATISTICS BY SPLITS:")
    for split_name, split_data in dataset.items():
        print(f"  {split_name.upper()}:")
        print(f"    - Number of samples: {len(split_data):,}")
        print(f"    - Number of meetings: {len(set(sample['meeting_id'] for sample in split_data))}")
        print(f"    - Unique speakers: {len(set(sample['speaker_id'] for sample in split_data))}")
    
    # Analysis by meetings
    print(f"\n🏢 MEETING ANALYSIS:")
    
    # Collect data from all splits
    all_meetings = defaultdict(list)
    all_speakers = set()
    all_durations = []
    all_audio_lengths = []
    
    for split_name, split_data in dataset.items():
        for sample in split_data:
            meeting_id = sample['meeting_id']
            speaker_id = sample['speaker_id']
            duration = sample['end_time'] - sample['begin_time']
            audio_length = len(sample['audio']['array'])
            
            all_meetings[meeting_id].append({
                'split': split_name,
                'speaker_id': speaker_id,
                'duration': duration,
                'audio_length': audio_length,
                'begin_time': sample['begin_time'],
                'end_time': sample['end_time']
            })
            all_speakers.add(speaker_id)
            all_durations.append(duration)
            all_audio_lengths.append(audio_length)
    
    # Meeting statistics
    meeting_stats = []
    for meeting_id, samples in all_meetings.items():
        speakers = set(s['speaker_id'] for s in samples)
        total_duration = sum(s['duration'] for s in samples)
        splits = set(s['split'] for s in samples)
        
        meeting_stats.append({
            'meeting_id': meeting_id,
            'num_samples': len(samples),
            'num_speakers': len(speakers),
            'speakers': sorted(speakers),
            'total_duration': total_duration,
            'splits': sorted(splits)
        })
    
    # Sort by number of samples
    meeting_stats.sort(key=lambda x: x['num_samples'], reverse=True)
    
    print(f"  - Total meetings: {len(meeting_stats)}")
    print(f"  - Total unique speakers: {len(all_speakers)}")
    
    # Top-10 meetings by number of samples
    print(f"\n  TOP-10 MEETINGS BY NUMBER OF SAMPLES:")
    for i, meeting in enumerate(meeting_stats[:10]):
        print(f"    {i+1:2d}. {meeting['meeting_id']}: {meeting['num_samples']:4d} samples, "
              f"{meeting['num_speakers']} speakers, {meeting['total_duration']:6.1f}s")
    
    # Audio duration statistics
    print(f"\n⏱️ AUDIO DURATION STATISTICS:")
    print(f"  - Minimum duration: {min(all_durations):.2f} seconds")
    print(f"  - Maximum duration: {max(all_durations):.2f} seconds")
    print(f"  - Average duration: {np.mean(all_durations):.2f} seconds")
    print(f"  - Median duration: {np.median(all_durations):.2f} seconds")
    print(f"  - Standard deviation: {np.std(all_durations):.2f} seconds")
    
    # Audio array length statistics
    print(f"\n🎵 AUDIO ARRAY LENGTH STATISTICS:")
    print(f"  - Minimum length: {min(all_audio_lengths):,} samples")
    print(f"  - Maximum length: {max(all_audio_lengths):,} samples")
    print(f"  - Average length: {np.mean(all_audio_lengths):,.0f} samples")
    print(f"  - Median length: {np.median(all_audio_lengths):,.0f} samples")
    
    # Distribution by number of speakers in meetings
    print(f"\n👥 DISTRIBUTION BY NUMBER OF SPEAKERS IN MEETINGS:")
    speaker_counts = Counter(meeting['num_speakers'] for meeting in meeting_stats)
    for num_speakers in sorted(speaker_counts.keys()):
        count = speaker_counts[num_speakers]
        print(f"  - {num_speakers} speakers: {count} meetings ({count/len(meeting_stats)*100:.1f}%)")
    
    # Speaker statistics
    print(f"\n🗣️ SPEAKER STATISTICS:")
    speaker_sample_counts = Counter()
    speaker_duration_counts = defaultdict(float)
    
    for meeting_id, samples in all_meetings.items():
        for sample in samples:
            speaker_sample_counts[sample['speaker_id']] += 1
            speaker_duration_counts[sample['speaker_id']] += sample['duration']
    
    # Top-10 speakers by number of samples
    print(f"  TOP-10 SPEAKERS BY NUMBER OF SAMPLES:")
    for i, (speaker_id, count) in enumerate(speaker_sample_counts.most_common(10)):
        duration = speaker_duration_counts[speaker_id]
        print(f"    {i+1:2d}. {speaker_id}: {count:4d} samples, {duration:6.1f}s")
    
    # Microphone statistics
    print(f"\n🎤 MICROPHONE STATISTICS:")
    mic_counts = Counter()
    for split_name, split_data in dataset.items():
        for sample in split_data:
            mic_counts[sample['microphone_id']] += 1
    
    for mic_id, count in mic_counts.most_common():
        print(f"  - {mic_id}: {count:,} samples")
    
    # Temporal statistics
    print(f"\n⏰ TEMPORAL STATISTICS:")
    all_begin_times = []
    all_end_times = []
    
    for split_name, split_data in dataset.items():
        for sample in split_data:
            all_begin_times.append(sample['begin_time'])
            all_end_times.append(sample['end_time'])
    
    print(f"  - Minimum start time: {min(all_begin_times):.1f} seconds")
    print(f"  - Maximum end time: {max(all_end_times):.1f} seconds")
    print(f"  - Total time range: {max(all_end_times) - min(all_begin_times):.1f} seconds")
    
    return {
        'meeting_stats': meeting_stats,
        'all_speakers': all_speakers,
        'all_durations': all_durations,
        'all_audio_lengths': all_audio_lengths,
        'speaker_sample_counts': speaker_sample_counts,
        'speaker_duration_counts': speaker_duration_counts,
        'mic_counts': mic_counts
    }

# Run analysis
stats = analyze_ami_dataset(dataset)


In [ ]:
# Comprehensive AMI Dataset Balance Analysis
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter, defaultdict
import numpy as np

def comprehensive_dataset_analysis(dataset):
    """
    Comprehensive AMI dataset balance analysis
    """
    print("="*80)
    print("📊 COMPREHENSIVE AMI DATASET BALANCE ANALYSIS")
    print("="*80)
    
    # Collect data from all splits
    all_data = []
    for split_name, split_data in dataset.items():
        for sample in split_data:
            all_data.append({
                'split': split_name,
                'meeting_id': sample['meeting_id'],
                'speaker_id': sample['speaker_id'],
                'microphone_id': sample['microphone_id'],
                'duration': sample['end_time'] - sample['begin_time'],
                'begin_time': sample['begin_time'],
                'end_time': sample['end_time'],
                'audio_length': len(sample['audio']['array'])
            })
    
    df = pd.DataFrame(all_data)
    
    # 1. Split distribution analysis
    print("\n📈 1. SPLIT DISTRIBUTION:")
    split_stats = df.groupby('split').agg({
        'meeting_id': 'nunique',
        'speaker_id': 'nunique', 
        'duration': ['count', 'sum', 'mean'],
        'audio_length': 'sum'
    }).round(2)
    
    print(split_stats)
    
    # 2. Meeting analysis
    print("\n🏢 2. MEETING ANALYSIS:")
    meeting_stats = df.groupby('meeting_id').agg({
        'speaker_id': 'nunique',
        'duration': ['count', 'sum', 'mean'],
        'split': lambda x: x.iloc[0],  # Meeting split
        'microphone_id': 'nunique'
    }).round(2)
    
    meeting_stats.columns = ['num_speakers', 'num_segments', 'total_duration', 'avg_segment_duration', 'split', 'num_mics']
    
    print(f"Total meetings: {len(meeting_stats)}")
    print(f"Average speakers per meeting: {meeting_stats['num_speakers'].mean():.1f}")
    print(f"Average segments per meeting: {meeting_stats['num_segments'].mean():.1f}")
    print(f"Average meeting duration: {meeting_stats['total_duration'].mean():.1f} seconds")
    
    # 3. Speaker analysis
    print("\n🗣️ 3. SPEAKER ANALYSIS:")
    speaker_stats = df.groupby('speaker_id').agg({
        'meeting_id': 'nunique',
        'duration': ['count', 'sum', 'mean'],
        'split': lambda x: list(set(x))
    }).round(2)
    
    speaker_stats.columns = ['num_meetings', 'num_segments', 'total_duration', 'avg_segment_duration', 'splits']
    
    print(f"Total unique speakers: {len(speaker_stats)}")
    print(f"Average meetings per speaker: {speaker_stats['num_meetings'].mean():.1f}")
    print(f"Average segments per speaker: {speaker_stats['num_segments'].mean():.1f}")
    print(f"Average total speech duration per speaker: {speaker_stats['total_duration'].mean():.1f} seconds")
    
    # 4. Microphone analysis
    print("\n🎤 4. MICROPHONE ANALYSIS:")
    mic_stats = df.groupby('microphone_id').agg({
        'duration': ['count', 'sum'],
        'meeting_id': 'nunique',
        'speaker_id': 'nunique'
    }).round(2)
    
    mic_stats.columns = ['num_segments', 'total_duration', 'num_meetings', 'num_speakers']
    print(mic_stats)
    
    # 5. Segment duration analysis
    print("\n⏱️ 5. SEGMENT DURATION ANALYSIS:")
    duration_stats = df['duration'].describe()
    print(duration_stats)
    
    # Check duration distribution
    short_segments = (df['duration'] < 1.0).sum()
    medium_segments = ((df['duration'] >= 1.0) & (df['duration'] < 5.0)).sum()
    long_segments = (df['duration'] >= 5.0).sum()
    
    print(f"\nDuration distribution:")
    print(f"  Short (< 1s): {short_segments:,} ({short_segments/len(df)*100:.1f}%)")
    print(f"  Medium (1-5s): {medium_segments:,} ({medium_segments/len(df)*100:.1f}%)")
    print(f"  Long (≥ 5s): {long_segments:,} ({long_segments/len(df)*100:.1f}%)")
    
    return df, meeting_stats, speaker_stats, mic_stats

# Run analysis
df, meeting_stats, speaker_stats, mic_stats = comprehensive_dataset_analysis(dataset)


In [ ]:
# Dataset Statistics Visualization
plt.style.use('default')
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('📊 AMI DATASET STATISTICS VISUALIZATION', fontsize=16, fontweight='bold')

# 1. Split distribution
ax1 = axes[0, 0]
split_counts = df['split'].value_counts()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
wedges, texts, autotexts = ax1.pie(split_counts.values, labels=split_counts.index, 
                                   autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Split Distribution', fontweight='bold')

# 2. Number of speakers in meetings
ax2 = axes[0, 1]
speaker_counts = meeting_stats['num_speakers'].value_counts().sort_index()
ax2.bar(speaker_counts.index, speaker_counts.values, color='#96CEB4', alpha=0.8)
ax2.set_xlabel('Number of Speakers')
ax2.set_ylabel('Number of Meetings')
ax2.set_title('Meeting Distribution by Number of Speakers', fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Segment duration
ax3 = axes[0, 2]
ax3.hist(df['duration'], bins=50, color='#FFB6C1', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Duration (seconds)')
ax3.set_ylabel('Number of Segments')
ax3.set_title('Segment Duration Distribution', fontweight='bold')
ax3.set_xlim(0, 10)  # Limit for better visibility
ax3.grid(True, alpha=0.3)

# 4. Meeting duration
ax4 = axes[1, 0]
ax4.hist(meeting_stats['total_duration'], bins=30, color='#DDA0DD', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Meeting Duration (seconds)')
ax4.set_ylabel('Number of Meetings')
ax4.set_title('Meeting Duration Distribution', fontweight='bold')
ax4.grid(True, alpha=0.3)

# 5. Segments per speaker
ax5 = axes[1, 1]
ax5.hist(speaker_stats['num_segments'], bins=30, color='#F0E68C', alpha=0.7, edgecolor='black')
ax5.set_xlabel('Number of Segments')
ax5.set_ylabel('Number of Speakers')
ax5.set_title('Segments per Speaker Distribution', fontweight='bold')
ax5.grid(True, alpha=0.3)

# 6. Microphone usage
ax6 = axes[1, 2]
mic_counts = df['microphone_id'].value_counts().sort_index()
bars = ax6.bar(mic_counts.index, mic_counts.values, color='#87CEEB', alpha=0.8)
ax6.set_xlabel('Microphone')
ax6.set_ylabel('Number of Segments')
ax6.set_title('Microphone Usage', fontweight='bold')
ax6.grid(True, alpha=0.3)

# Add values on bars
for bar in bars:
    height = bar.get_height()
    ax6.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
             f'{int(height):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Additional plots
fig2, axes2 = plt.subplots(1, 2, figsize=(15, 6))
fig2.suptitle('📈 ADDITIONAL ANALYSIS', fontsize=16, fontweight='bold')

# Correlation between meeting characteristics
ax7 = axes2[0]
scatter = ax7.scatter(meeting_stats['num_speakers'], meeting_stats['total_duration'], 
                     c=meeting_stats['num_segments'], cmap='viridis', alpha=0.6, s=50)
ax7.set_xlabel('Number of Speakers')
ax7.set_ylabel('Meeting Duration (sec)')
ax7.set_title('Speakers vs Meeting Duration\n(color = number of segments)', fontweight='bold')
ax7.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax7, label='Number of Segments')

# Top-10 speakers by number of segments
ax8 = axes2[1]
top_speakers = speaker_stats.nlargest(10, 'num_segments')
bars = ax8.barh(range(len(top_speakers)), top_speakers['num_segments'], color='#98FB98', alpha=0.8)
ax8.set_yticks(range(len(top_speakers)))
ax8.set_yticklabels(top_speakers.index, fontsize=9)
ax8.set_xlabel('Number of Segments')
ax8.set_title('Top-10 Speakers by Number of Segments', fontweight='bold')
ax8.grid(True, alpha=0.3)

# Add values on bars
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax8.text(width + width*0.01, bar.get_y() + bar.get_height()/2,
             f'{int(width)}', ha='left', va='center', fontsize=8)

plt.tight_layout()
plt.show()


# 🎯 DATA SPLITTING STRATEGIES FOR FEDERATED LEARNING

## 📊 AMI Dataset Characteristics Analysis:

### Key Features:
- **171 meetings** with **190 unique speakers**
- **134,243 audio segments** with total duration ~96 hours
- **Uneven distribution**: some speakers appear in multiple meetings, others only once
- **Overlapping speech**: ~15.6% of time contains speaker overlaps
- **Different microphones**: H00-H04 with varying recording quality

### Challenges for Federated Learning:
1. **Data imbalance**: different clients will receive different amounts of data
2. **Speaker overlaps**: one speaker may appear in multiple clients' data
3. **Quality variations**: different microphones provide different audio quality
4. **Temporal dependencies**: segments from the same meeting may be related

## 🎯 Recommended Splitting Strategies:


In [ ]:
# Analysis of Data Splitting Strategies for Federated Learning
def analyze_client_split_strategies(df, meeting_stats, speaker_stats, num_clients=5):
    """
    Analyzes various data splitting strategies for federated learning
    """
    print("="*80)
    print("🎯 FEDERATED LEARNING DATA SPLITTING STRATEGIES ANALYSIS")
    print("="*80)
    
    strategies = {}
    
    # 1. STRATEGY: Meeting-based splitting
    print("\n📋 1. STRATEGY: MEETING-BASED SPLITTING")
    print("   Principle: Each client gets complete meetings")
    
    meetings = df['meeting_id'].unique()
    np.random.seed(42)  # For reproducibility
    np.random.shuffle(meetings)
    
    meetings_per_client = len(meetings) // num_clients
    meeting_clients = {}
    
    for i in range(num_clients):
        start_idx = i * meetings_per_client
        if i == num_clients - 1:  # Last client gets all remaining meetings
            end_idx = len(meetings)
        else:
            end_idx = (i + 1) * meetings_per_client
        
        client_meetings = meetings[start_idx:end_idx]
        meeting_clients[f'client_{i}'] = client_meetings
        
        # Client statistics
        client_data = df[df['meeting_id'].isin(client_meetings)]
        print(f"   Client {i}: {len(client_meetings)} meetings, {len(client_data)} segments, "
              f"{client_data['speaker_id'].nunique()} speakers, {client_data['duration'].sum():.1f}s")
    
    strategies['meeting_based'] = meeting_clients
    
    # 2. STRATEGY: Speaker-based splitting
    print("\n📋 2. STRATEGY: SPEAKER-BASED SPLITTING")
    print("   Principle: Each client gets unique speakers")
    
    speakers = df['speaker_id'].unique()
    np.random.shuffle(speakers)
    
    speakers_per_client = len(speakers) // num_clients
    speaker_clients = {}
    
    for i in range(num_clients):
        start_idx = i * speakers_per_client
        if i == num_clients - 1:
            end_idx = len(speakers)
        else:
            end_idx = (i + 1) * speakers_per_client
        
        client_speakers = speakers[start_idx:end_idx]
        speaker_clients[f'client_{i}'] = client_speakers
        
        # Client statistics
        client_data = df[df['speaker_id'].isin(client_speakers)]
        print(f"   Client {i}: {len(client_speakers)} speakers, {len(client_data)} segments, "
              f"{client_data['meeting_id'].nunique()} meetings, {client_data['duration'].sum():.1f}s")
    
    strategies['speaker_based'] = speaker_clients
    
    # 3. STRATEGY: Random segment splitting
    print("\n📋 3. STRATEGY: RANDOM SEGMENT SPLITTING")
    print("   Principle: Each segment is randomly assigned to a client")
    
    segments = df.index.tolist()
    np.random.shuffle(segments)
    
    segments_per_client = len(segments) // num_clients
    random_clients = {}
    
    for i in range(num_clients):
        start_idx = i * segments_per_client
        if i == num_clients - 1:
            end_idx = len(segments)
        else:
            end_idx = (i + 1) * segments_per_client
        
        client_segments = segments[start_idx:end_idx]
        random_clients[f'client_{i}'] = client_segments
        
        # Client statistics
        client_data = df.loc[client_segments]
        print(f"   Client {i}: {len(client_segments)} segments, "
              f"{client_data['speaker_id'].nunique()} speakers, {client_data['meeting_id'].nunique()} meetings, "
              f"{client_data['duration'].sum():.1f}s")
    
    strategies['random_segments'] = random_clients
    
    # 4. STRATEGY: Microphone-based splitting
    print("\n📋 4. STRATEGY: MICROPHONE-BASED SPLITTING")
    print("   Principle: Each client gets data from specific microphones")
    
    microphones = df['microphone_id'].unique()
    mic_clients = {}
    
    for i in range(num_clients):
        if i < len(microphones):
            client_mics = [microphones[i]]
        else:
            # If more clients than microphones, distribute evenly
            client_mics = [microphones[i % len(microphones)]]
        
        mic_clients[f'client_{i}'] = client_mics
        
        # Client statistics
        client_data = df[df['microphone_id'].isin(client_mics)]
        print(f"   Client {i}: microphones {client_mics}, {len(client_data)} segments, "
              f"{client_data['speaker_id'].nunique()} speakers, {client_data['meeting_id'].nunique()} meetings, "
              f"{client_data['duration'].sum():.1f}s")
    
    strategies['microphone_based'] = mic_clients
    
    # 5. STRATEGY: Balanced splitting
    print("\n📋 5. STRATEGY: BALANCED SPLITTING")
    print("   Principle: Aim for even distribution across all metrics")
    
    # Group by meetings and sort by size
    meeting_sizes = df.groupby('meeting_id').size().sort_values(ascending=False)
    
    # Distribute meetings among clients, trying to balance size
    client_meetings = {f'client_{i}': [] for i in range(num_clients)}
    client_sizes = [0] * num_clients
    
    for meeting_id, size in meeting_sizes.items():
        # Assign meeting to client with smallest current size
        min_client = client_sizes.index(min(client_sizes))
        client_meetings[f'client_{min_client}'].append(meeting_id)
        client_sizes[min_client] += size
    
    balanced_clients = {}
    for i in range(num_clients):
        client_meetings_list = client_meetings[f'client_{i}']
        balanced_clients[f'client_{i}'] = client_meetings_list
        
        # Client statistics
        client_data = df[df['meeting_id'].isin(client_meetings_list)]
        print(f"   Client {i}: {len(client_meetings_list)} meetings, {len(client_data)} segments, "
              f"{client_data['speaker_id'].nunique()} speakers, {client_data['duration'].sum():.1f}s")
    
    strategies['balanced'] = balanced_clients
    
    return strategies

# Analyze strategies
strategies = analyze_client_split_strategies(df, meeting_stats, speaker_stats, num_clients=5)
